In [ ]:
import torch
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, Qwen3VLProcessor
import re
from qwen_vl_utils import process_vision_info
from transformers.generation.logits_process import LogitsProcessorList
from transformers import AutoProcessor
import json

In [ ]:
# 指向你本地的模型目录
model_path = "qwen3_4b_vl"
device="cuda"
dtype=torch.bfloat16
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=dtype,
    device_map=device,
    attn_implementation="flash_attention_2", 
    trust_remote_code=True
).eval()

In [ ]:
TEST_IMG = "/home/s414f/s414f1/414f/ano_vlms/Qwen3-VL-main/evaluation/ODinW-13/data/bottle1/images/test_contamination_003.png"
prompt = (
      "Classify the image into one of the following categories: good, broken_large, broken_small, contamination."
      "If the image is not classified as good, provide the defect location (bounding box), cause analysis, and corresponding improvement measures."
      "Output the detection result in the following JSON format:{type: ..., bbox: [...], cause_analysis: ..., solution: ..., think: ...}"
      "If the image is good, leave the fields bbox, cause_analysis, and solution empty."
      
)
messages = [{
            "role": "user", 
             "content": [
                {"type": "image", "image": TEST_IMG},
                {"type": "text", "text": prompt}
]}]

text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, _ = process_vision_info(messages)
inputs = processor(text=[text_prompt], images=image_inputs, return_tensors="pt").to(device)
output_ids = model.generate(**inputs, max_new_tokens=2048, use_cache=True)

generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
raw_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:
inputs.input_ids

In [ ]:
def extract_json_from_vl_output(text):
    """
    从Qwen3-VL模型的输出文本中提取并重新组织JSON数据
    
    Args:
        text: 模型输出的完整文本
    
    Returns:
        list: 重新组织后的JSON数据列表，格式为：
              [{"type": "...", "bbox": [...], "cause_analysis": "...", "solution": "...", "think": "..."}]
    """
    
    # 1. 提取think内容（<think>和</think>之间的所有内容）
    think_pattern = r'(.*?)</think>'
    think_matches = re.findall(think_pattern, text, re.DOTALL)
    
    think_content = ""
    if think_matches:
        # 取最后一个<think>内容
        think_content = think_matches[-1].strip()
    else:
        # 如果没有明确的think标签，尝试提取推理部分
        # 从"So, let's look"开始到JSON部分之前
        think_start_pattern = r'So,\s*let\'s\s*look.*?(?=\[|\{)'
        think_match = re.search(think_start_pattern, text, re.DOTALL | re.IGNORECASE)
        if think_match:
            think_content = think_match.group(0).strip()
    
    # 2. 提取JSON数据
    # 首先尝试找到最后一个JSON数组
    json_pattern = r'\[\s*\{.*?\}\s*\]'
    json_matches = re.findall(json_pattern, text, re.DOTALL)
    
    extracted_data = []
    
    if json_matches:
        # 通常最后一个匹配是最终的JSON输出
        json_str = json_matches[-1]
        
        try:
            # 解析JSON
            parsed_json = json.loads(json_str)
            
            if isinstance(parsed_json, list):
                for item in parsed_json:
                    # 重新组织为指定格式
                    new_item = {}
                    
                    # type字段：优先使用"defect"，如果没有则尝试其他可能的字段
                    if "label" in item:
                        new_item["type"] = item["label"]
                    elif "type" in item:
                        new_item["type"] = item["type"]
                    elif "category" in item:
                        new_item["type"] = item["category"]
                    else:
                        new_item["type"] = "unknown"
                    
                    # bbox字段：确保格式正确
                    if "bbox" in item:
                        # 如果bbox是字符串，尝试解析
                        if isinstance(item["bbox"], str):
                            try:
                                # 处理"[x1, y1, x2, y2]"格式
                                bbox_str = item["bbox"].strip("[]")
                                bbox_list = [int(float(x.strip())) for x in bbox_str.split(",")]
                                new_item["bbox"] = bbox_list
                            except:
                                new_item["bbox"] = []
                        elif isinstance(item["bbox"], list):
                            new_item["bbox"] = item["bbox"]
                        else:
                            new_item["bbox"] = []
                    else:
                        new_item["bbox"] = []
                    
                    # cause_analysis字段
                    if "cause_analysis" in item:
                        new_item["cause_analysis"] = item["cause_analysis"]
                    elif "analysis" in item:
                        new_item["cause_analysis"] = item["analysis"]
                    else:
                        new_item["cause_analysis"] = ""
                    
                    # solution字段
                    if "solution" in item:
                        new_item["solution"] = item["solution"]
                    elif "recommendation" in item:
                        new_item["solution"] = item["recommendation"]
                    else:
                        new_item["solution"] = ""
                    
                    # think字段
                    new_item["think"] = think_content
                    
                    extracted_data.append(new_item)
            
        except json.JSONDecodeError as e:
            print(f"JSON解析错误: {e}")
            print(f"原始JSON字符串: {json_str[:100]}...")
    
    return extracted_data

def process_vl_output_file(input_text, output_file=None):
    """
    处理VL模型输出文本并保存结果
    
    Args:
        input_text: 输入文本或文件路径
        output_file: 输出文件路径（可选）
    
    Returns:
        list: 提取的数据
    """
    # 如果input_text是文件路径，读取文件内容
    if isinstance(input_text, str) and input_text.endswith(('.txt', '.json', '.log')):
        try:
            with open(input_text, 'r', encoding='utf-8') as f:
                text_content = f.read()
        except FileNotFoundError:
            print(f"文件不存在: {input_text}")
            return []
    else:
        text_content = input_text
    
    # 提取数据
    extracted_data = extract_json_from_vl_output(text_content)
    
    # 输出结果
    if extracted_data:
        print(f"成功提取 {len(extracted_data)} 个数据项")
        
        # 格式化输出
        formatted_json = json.dumps(extracted_data, indent=2, ensure_ascii=False)
        print("\n提取的数据:")
        print(formatted_json)
        
        # 保存到文件
        if output_file:
            try:
                with open(output_file, 'w', encoding='utf-8') as f:
                    f.write(formatted_json)
                print(f"\n结果已保存到: {output_file}")
            except Exception as e:
                print(f"保存文件失败: {e}")
    else:
        print("未找到有效的JSON数据")
    
    return extracted_data

In [ ]:
result = process_vl_output_file(raw_text, "extracted_result.json")
raw_text